# 文本预处理


## 环境配置


In [ ]:
import torch
from src.utils import (
    read_time_machine, tokenize, Vocab,
    load_corpus_time_machine
)

## 练习8.2.1

**题目：** 词元化是一个关键的预处理步骤，它因语言而异。尝试找到另外三种常用的词元化文本的方法。

**解答：**

**1. BPE (Byte-Pair Encoding) —— 字节对编码**

BPE 是一种自底向上的子词构建方法，本质是贪心算法。具体流程为：

(1) 确定词表大小（即子词总数量），作为算法终止条件。

(2) 统计语料库中每个单词出现的频率，并在每个单词末尾添加结束符 `</w>`（用于标识单词边界，避免将不同单词的子词混淆）。

(3) 将每个单词拆分为单个字符（初始子词），统计每个字符的出现频率，构建初始子词词表。

(4) 统计文本中所有相邻子词对出现的频数，合并出现频数最高的子词对，生成新的子词并更新词表（例如 "e"+"s" → "es"）。

(5) 重复步骤 (4)，直到达到步骤 (1) 确定的词表大小或合并次数上限。

优点：通过高频子词的合并，实现对语料库的数据压缩，用最少的子词表示语料库；能够处理未登录词（OOV）问题，将罕见词分解为已知子词。

**2. WordPiece**

WordPiece 与 BPE 的子词构建过程类似，但合并策略不同。具体流程为：

(1) 与 BPE 类似，先构建初始词表（所有字符 + 特殊符号，如 [UNK]、[CLS]、[SEP] 等）。

(2) 不同于 BPE 选择出现频数最高的子词对，WordPiece 选择能够最大化训练数据语言模型似然值的子词对进行合并。

(3) 似然值提升量的计算方式为：该子词对出现的频数除以第一个子词的频率与第二个子词频率的乘积。公式为：score = count(pair) / (count(first) × count(second))。这实际上衡量了子词对中两个子词之间的"粘合度"——两个子词之间的关联性越强，score 越大。

(4) 选择使似然值提升最大的子词对进行合并，重复直到达到预定词表大小。

优点：更关注语义层面的关联性（而非单纯的频数），生成的子词表通常更具语言学意义。BERT 等预训练模型使用 WordPiece 作为分词器。

**3. SentencePiece**

SentencePiece 是一种语言无关的子词词元化工具，不依赖于特定语言的分词规则。其核心特点为：

- 将空格也视为普通字符进行编码，不区分单词边界。这样做的好处是不需要预先对文本进行分词（避免了对特定语言分词器的依赖）。
- 将整个输入文本（整个句子或文档）视为一个连续的 Unicode 字符序列，然后通过 BPE 或 Unigram 语言模型的方式构造子词词表。

SentencePiece 特别适用于中文、日文、韩文（CJK）等不使用空格分隔单词的语言，也适用于需要处理多种语言的统一模型（多语言翻译、多语言预训练等）。Google 的 T5 和 XLNet 模型使用了 SentencePiece。

优点：语言无关，无需预处理分词步骤，训练和推理完全一致。



## 练习8.2.2

**题目：** 在本节的实验中，将文本词元为单词和更改 Vocab 实例的 min_freq 参数。这对词表大小有何影响？

**解答：**

当词元设置为 Char 时，我们将文本序列中的每个字符视为一个词元。由于字符的种类数量有限（字母、数字、标点符号等），词表大小通常较小（本节实验为 28）。但相应地，生成的语料库长度会增大（每个字符对应一个索引，本节实验为 170580），这使得模型需要处理更长的序列，带来更大的计算和内存开销。

当词元设置为 Word 时，文本按空格和标点符号进行分割，每个单词视为一个词元。由于语言的单词种类丰富，词表大小会显著增大（本节实验为 4580），但语料库长度会大幅缩短（本节实验为 32775），因为每个时间步表示一个完整单词而非单个字符。词表增大意味着输出层参数量增大、需要更多训练数据来学习每个词的表示。

对比分析：Char 模式词表小（28）、序列长（170580）——输出层参数少，但序列建模困难；Word 模式词表大（4580）、序列短（32775）——序列建模容易，但输出层参数多。词表越大，计算资源需求越大；序列越长，梯度传播越困难。

min_freq 参数主要用来实现对低频词的过滤。将出现次数低于 min_freq 的词（即低频词）全部映射为 `<unk>` 标记（索引 0），从而缩减词表大小。这是词表构建中的一个重要超参数：min_freq 增大 → 词表缩小 → 输出层参数减少 → 计算效率提升；但同时低频词信息丢失 → 可能影响模型理解能力（尤其对专有名词等）。在实际应用中需要根据语料库大小、任务需求和计算资源进行权衡选择。



以下使用 torch 编程验证：


In [ ]:
# char 模式：将文本切分为单个字符作为词元
corpus, vocab = load_corpus_time_machine()
print(f'char 模式: corpus={len(corpus)}, vocab={len(vocab)}')
# 输出: corpus=170580, vocab=28

# word 模式：按空格和标点符号切分为单词
lines = read_time_machine()
tokens = tokenize(lines, 'word')
vocab_word = Vocab(tokens)
corpus_word = [vocab_word[token] for line in tokens for token in line]
print(f'word 模式: corpus={len(corpus_word)}, vocab={len(vocab_word)}')
# 输出: corpus=32775, vocab=4580

# min_freq 参数：控制低频词的过滤阈值
# min_freq=0：不过滤任何低频词
vocab0 = Vocab(tokens, min_freq=0)
print(f'min_freq=0 词表大小: {len(vocab0)}')
# 输出: 4580

# min_freq=10：将出现次数 <10 的低频词映射为 <unk>
vocab10 = Vocab(tokens, min_freq=10)
print(f'min_freq=10 词表大小: {len(vocab10)}')
# 输出: 远小于 4580
# 可见 min_freq 增大后，词表大小显著缩小，低频词被映射为 <unk>


---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)

